# Zn Expert vs Full (9-Element) Pipeline — Figshare Only

This notebook trains two pipeline variants sequentially and compares them on Zn test data, using **Figshare data only** (no MaterialsCloud/OmniXAS raw data).

1. **Expert Zn run:** trains encoder, UniversalXAS, and tuned heads on Zn data only.
2. **Full 9-element run:** trains encoder and UniversalXAS on all 8 OmniXAS elements + Zn, then fine-tunes tuned heads for all 9 elements.

The 8 existing element splits are **read in code only** — nothing is copied or modified. Zn splits are newly written. A combined dataset directory is created with symlinks to the existing 8-element files and new Zn files.

Hyperparameters (match unscaled Figshare reference):
- alpha (`universal-loss-balance-power`): 1.5
- UniversalXAS learning rate: 0.002, seed: 43
- Tuned learning rate: 0.001, seed: 42
- Dropout: 0.05
- Encoder rows per element: 64
- Batch size: 4096

Both training processes run sequentially in the notebook process, so each pipeline's tqdm progress bars stream directly into the cell output. Set `CUDA_VISIBLE_DEVICES` before starting Jupyter (e.g. `CUDA_VISIBLE_DEVICES=7 jupyter notebook`). This notebook does not select or inspect GPUs.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

def find_repo_root():
    cwd = Path.cwd().resolve()
    if (cwd / 'tutorial_omnixas').is_dir():
        return cwd
    if cwd.name == 'tutorial_omnixas' and (cwd.parent / 'omnixas').is_dir():
        return cwd.parent
    raise FileNotFoundError('Start Jupyter from the OmniXAS repository root or tutorial_omnixas directory.')

REPO_ROOT = find_repo_root()
DATA_ROOT = Path(os.environ.get('OMNIXAS_DATA_ROOT', REPO_ROOT.parent / 'OmniXAS_data')).expanduser()
RAW_ROOT = DATA_ROOT / 'figshare'
NEW_ROOT = DATA_ROOT / 'figshare_141_splits_unscaled'
COMBINED_DATA_ROOT = DATA_ROOT / 'figshare_141_splits_unscaled_zn9'
OUTPUT_ROOT = REPO_ROOT / 'output' / 'training' / 'zn_expert_study'
ANALYSIS_DIR = REPO_ROOT / 'reports' / 'zn_expert_study'
SCRIPT = REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py'

ELEMENTS = ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu']
ZN_ELEMENT = 'Zn'
ALL_ELEMENTS = ELEMENTS + [ZN_ELEMENT]
SPLITS = ['train', 'val', 'test']

# Hyperparameters (match unscaled Figshare reference)
SEED = 42
HEAD_DROPOUT = 0.05
UNIVERSAL_LOSS_BALANCE_POWER = 1.5
UNIVERSAL_LEARNING_RATE = 0.002
UNIVERSAL_SEED = 43
TUNED_LEARNING_RATE = 0.001
TUNED_SEED = 42

ZN_K_EDGE_START_EV = 9659.0  # LBL X-ray Data Booklet Table 1-1a
ZN_WINDOW_EV = 35.0
ZN_POINT_COUNT = 141
ZN_STEP_EV = 0.25

required = [
    REPO_ROOT / 'models' / 'M3GNet-MP-2021.2.8-PES' / 'model.pt',
    REPO_ROOT / 'omnixas' / 'featurizer' / 'm3gnet_featurizer.py',
    SCRIPT,
    RAW_ROOT / 'manifest.csv',
    NEW_ROOT / 'manifest.csv',
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f'Required input is missing: {path}')

print(f'Repository: {REPO_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Existing Figshare splits (read only): {NEW_ROOT}')
print(f'Combined dataset (new Zn + symlinks): {COMBINED_DATA_ROOT}')
print(f'Output root: {OUTPUT_ROOT}')
print(f'Zn K-edge start: {ZN_K_EDGE_START_EV} eV')

In [ ]:
def read_ids(path):
    rows = [line.strip() for line in path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
    if not rows:
        raise ValueError(f'Empty ID split: {path}')
    return rows

def read_targets(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing target file: {path}')
    values = np.loadtxt(path, dtype=np.float32, ndmin=2)
    if values.shape[1] != 141 or not np.isfinite(values).all():
        raise ValueError(f'Targets must be finite with 141 columns: {path} {values.shape}')
    if (values < 0).any():
        raise ValueError(f'Targets must be nonnegative: {path}')
    return values

def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(i + '\n' for i in ids), encoding='utf-8')

def write_targets(path, values):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(path, values, fmt='%.9g')

def interpolate_spectrum(energy, mu, grid_start, grid_end, n_points):
    mask = np.isfinite(energy) & np.isfinite(mu) & (mu > 0)
    energy, mu = energy[mask], mu[mask]
    if len(energy) < 2:
        raise ValueError('Not enough finite positive samples')
    order = np.argsort(energy)
    energy, mu = energy[order], mu[order]
    grid = np.linspace(grid_start, grid_end, n_points, dtype=np.float64)
    return np.interp(grid, energy, mu)

def load_zn_manifest():
    manifest_path = RAW_ROOT / 'manifest.csv'
    df = pd.read_csv(manifest_path, dtype={'element': str})
    zn = df[df['element'] == 'Zn'].copy()
    zn['site'] = pd.to_numeric(zn['site'], errors='raise').astype(int)
    return zn

zn_manifest = load_zn_manifest()
print(f'Zn records in Figshare manifest: {len(zn_manifest)}')

# Verify Zn POSCAR files exist (read-only check)
missing_poscar = []
for _, row in zn_manifest.iterrows():
    spectrum_path = RAW_ROOT / row['site_directory'] / 'spectrum_raw.dat'
    poscar_path = RAW_ROOT / row['site_directory'] / 'POSCAR'
    if not spectrum_path.is_file():
        missing_poscar.append(f'spectrum: {spectrum_path}')
    if not poscar_path.is_file():
        missing_poscar.append(f'POSCAR: {poscar_path}')
if missing_poscar:
    raise FileNotFoundError(f'Missing Zn source files:\n' + '\n'.join(missing_poscar[:10]))
print(f'All {len(zn_manifest)} Zn records have spectrum_raw.dat and POSCAR')

In [ ]:
# Read existing 8-element Figshare splits in code (read-only, no copy, no modify)
existing_counts = {}
for element in ELEMENTS:
    element_counts = {}
    for split in SPLITS:
        ids_path = NEW_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt'
        y_path = NEW_ROOT / 'ml_data' / f'{element}_FEFF_{split}_y.txt'
        ids = read_ids(ids_path)
        y = read_targets(y_path)
        if len(ids) != len(y):
            raise ValueError(f'Row mismatch for {element} {split}: {len(ids)} != {len(y)}')
        element_counts[split] = len(ids)
    existing_counts[element] = element_counts
    
    counts_str = ', '.join(f'{s}={c}' for s, c in element_counts.items())
    
    print(f'{element}: {counts_str}')

total_existing = sum(sum(v.values()) for v in existing_counts.values())
print(f'Total existing 8-element spectra: {total_existing}')
print('Existing splits read in code only — nothing copied or modified.')

In [ ]:
# Create material-level Zn split (80/10/10) — new data only
np.random.default_rng(SEED)
zn_materials = np.asarray(sorted(zn_manifest['material_id'].unique()), dtype=object)
np.random.default_rng(SEED).shuffle(zn_materials)
n_materials = len(zn_materials)
n_train = max(1, int(round(0.80 * n_materials)))
n_val = max(1, int(round(0.10 * n_materials))) if n_materials >= 3 else 0
n_test = n_materials - n_train - n_val
if n_test < 1:
    n_test = 1
    n_train = n_materials - n_val - n_test

zn_train_materials = set(zn_materials[:n_train])
zn_val_materials = set(zn_materials[n_train:n_train + n_val])
zn_test_materials = set(zn_materials[n_train + n_val:])

print(f'Zn materials: train={len(zn_train_materials)}, val={len(zn_val_materials)}, test={len(zn_test_materials)}')

# Build energy grid for Zn targets (9659 eV K-edge start from LBL Table 1-1a)
zn_grid_start = ZN_K_EDGE_START_EV
zn_grid_end = ZN_K_EDGE_START_EV + ZN_WINDOW_EV
zn_energy_grid = np.linspace(zn_grid_start, zn_grid_end, ZN_POINT_COUNT, dtype=np.float64)

zn_splits = {}
for split, material_set in (('train', zn_train_materials), ('val', zn_val_materials), ('test', zn_test_materials)):
    subset = zn_manifest[zn_manifest['material_id'].isin(material_set)].copy()
    ids = []
    targets = []
    for _, row in subset.iterrows():
        material_id = row['material_id']
        site = int(row['site'])
        ids.append(f'{material_id}_{site:03d}')
        spectrum_path = RAW_ROOT / row['site_directory'] / 'spectrum_raw.dat'
        table = np.loadtxt(spectrum_path, comments='#', dtype=np.float64, ndmin=2)
        energy = table[:, 0]
        mu = table[:, 3]
        y = interpolate_spectrum(energy, mu, zn_grid_start, zn_grid_end, ZN_POINT_COUNT)
        targets.append(y)
    targets = np.array(targets, dtype=np.float32)
    zn_splits[split] = {'ids': ids, 'y': targets}
    print(f'Zn {split}: {len(ids)} spectra, targets shape {targets.shape}')

# Verify no material leakage
all_materials = zn_train_materials | zn_val_materials | zn_test_materials
assert len(all_materials) == n_materials, 'Material split does not cover all materials'
assert not (zn_train_materials & zn_val_materials), 'Train/val leakage'
assert not (zn_train_materials & zn_test_materials), 'Train/test leakage'
assert not (zn_val_materials & zn_test_materials), 'Val/test leakage'
print('No material leakage in Zn split')

In [ ]:
# Create combined dataset directory:
# - 8 existing elements: symlinks to the read-only Figshare files (nothing copied or modified)
# - Zn: newly written ID and target files

if COMBINED_DATA_ROOT.exists():
    raise FileExistsError(f'Combined dataset already exists: {COMBINED_DATA_ROOT}')

COMBINED_DATA_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('material_id_and_site', 'ml_data'):
    (COMBINED_DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Symlink existing 8-element files (read-only references)
for element in ELEMENTS:
    for split in SPLITS:
        src_ids = NEW_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt'
        src_y = NEW_ROOT / 'ml_data' / f'{element}_FEFF_{split}_y.txt'
        dst_ids = COMBINED_DATA_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt'
        dst_y = COMBINED_DATA_ROOT / 'ml_data' / f'{element}_FEFF_{split}_y.txt'
        if not dst_ids.exists():
            dst_ids.symlink_to(src_ids)
        if not dst_y.exists():
            dst_y.symlink_to(src_y)
    print(f'Symlinked {element} (read-only, no copy)')

# Write new Zn files only
for split in SPLITS:
    write_ids(COMBINED_DATA_ROOT / 'material_id_and_site' / f'Zn_FEFF_{split}.txt', zn_splits[split]['ids'])
    write_targets(COMBINED_DATA_ROOT / 'ml_data' / f'Zn_FEFF_{split}_y.txt', zn_splits[split]['y'])
    print(f'Wrote Zn {split}: {len(zn_splits[split]["ids"])} spectra')

# Validate combined dataset row counts and no leakage (read-only checks)
for element in ALL_ELEMENTS:
    material_sets = {}
    for split in SPLITS:
        ids = read_ids(COMBINED_DATA_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt')
        y = read_targets(COMBINED_DATA_ROOT / 'ml_data' / f'{element}_FEFF_{split}_y.txt')
        if len(ids) != len(y):
            raise ValueError(f'Row mismatch for {element} {split}: {len(ids)} != {len(y)}')
        material_sets[split] = {value.rsplit('_', 1)[0] for value in ids}
    for left_index, split in enumerate(SPLITS):
        for other in SPLITS[left_index + 1:]:
            overlap = material_sets[split] & material_sets[other]
            if overlap:
                raise ValueError(f'Material leakage for {element}: {split}/{other}: {sorted(overlap)[:5]}')
    counts = [len(read_ids(COMBINED_DATA_ROOT / 'material_id_and_site' / f'{element}_FEFF_{split}.txt')) for split in SPLITS]
    print(f'{element}: train={counts[0]}, val={counts[1]}, test={counts[2]}')

print(f'Combined dataset ready: {COMBINED_DATA_ROOT}')
print('8 existing elements: symlinks (read-only). Zn: new files only.')

## Train Expert Zn Run

Trains the scratch M3GNet encoder, exports features, trains UniversalXAS (Zn only), and trains the tuned Zn head. Uses `runpy.run_path` so tqdm progress bars stream directly into this cell output.

The pipeline module's `FEFF_TASKS` is temporarily set to `['Zn_FEFF']` so the entire pipeline runs on Zn data only.

In [ ]:
import train_m3gnet_xas_pipeline as pipeline

ZN_EXPERT_RUN_NAME = 'zn_expert'
ZN_EXPERT_RUN_DIR = OUTPUT_ROOT / ZN_EXPERT_RUN_NAME

if ZN_EXPERT_RUN_DIR.exists():
    raise FileExistsError(f'Expert run directory already exists: {ZN_EXPERT_RUN_DIR}')

original_feff_tasks = pipeline.FEFF_TASKS
pipeline.FEFF_TASKS = ['Zn_FEFF']

zn_training_args = [
    str(SCRIPT),
    '--dataset-root', str(COMBINED_DATA_ROOT),
    '--raw-root', str(RAW_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--run-name', ZN_EXPERT_RUN_NAME,
    '--seed', str(SEED),
    '--precision', 'bf16-mixed',
    '--encoder-rows-per-element', '64',
    '--encoder-eval-batch-size', '1024',
    '--num-workers', '32',
    '--batch-size', '4096',
    '--head-dropout', str(HEAD_DROPOUT),
    '--universal-loss-balance-power', str(UNIVERSAL_LOSS_BALANCE_POWER),
    '--universal-lr', str(UNIVERSAL_LEARNING_RATE),
    '--universal-seed', str(UNIVERSAL_SEED),
    '--universal-es-metric', 'macro_relative_median',
    '--tuned-lr', str(TUNED_LEARNING_RATE),
    '--tuned-seed', str(TUNED_SEED),
    '--tuned-schedule', 'plateau',
    '--tuned-loss-normalize',
]

previous_argv = sys.argv
sys.argv = zn_training_args
try:
    print('Starting expert Zn pipeline training...')
    pipeline.main()
    print('Expert Zn pipeline training complete.')
finally:
    sys.argv = previous_argv
    pipeline.FEFF_TASKS = original_feff_tasks

if not (ZN_EXPERT_RUN_DIR / 'RUN_COMPLETE.json').is_file():
    raise FileNotFoundError(f'Expert run did not complete: missing {ZN_EXPERT_RUN_DIR / "RUN_COMPLETE.json"}')
print(f'Expert Zn run complete: {ZN_EXPERT_RUN_DIR}')

## Train Full 9-Element Run

Trains the scratch M3GNet encoder on all 8 OmniXAS elements + Zn, exports features, trains UniversalXAS on all 9 elements, and fine-tunes tuned heads for all 9 elements. Uses `runpy.run_path` so tqdm progress bars stream directly into this cell output.

The pipeline module's `FEFF_TASKS` is temporarily set to all 9 tasks.

In [ ]:
ZN_FULL_RUN_NAME = 'zn_full'
ZN_FULL_RUN_DIR = OUTPUT_ROOT / ZN_FULL_RUN_NAME

if ZN_FULL_RUN_DIR.exists():
    raise FileExistsError(f'Full run directory already exists: {ZN_FULL_RUN_DIR}')

original_feff_tasks = pipeline.FEFF_TASKS
zn_full_tasks = [f'{e}_FEFF' for e in ALL_ELEMENTS]
pipeline.FEFF_TASKS = zn_full_tasks

zn_full_training_args = [
    str(SCRIPT),
    '--dataset-root', str(COMBINED_DATA_ROOT),
    '--raw-root', str(RAW_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--run-name', ZN_FULL_RUN_NAME,
    '--seed', str(SEED),
    '--precision', 'bf16-mixed',
    '--encoder-rows-per-element', '64',
    '--encoder-eval-batch-size', '1024',
    '--num-workers', '32',
    '--batch-size', '4096',
    '--head-dropout', str(HEAD_DROPOUT),
    '--universal-loss-balance-power', str(UNIVERSAL_LOSS_BALANCE_POWER),
    '--universal-lr', str(UNIVERSAL_LEARNING_RATE),
    '--universal-seed', str(UNIVERSAL_SEED),
    '--universal-es-metric', 'macro_relative_median',
    '--tuned-lr', str(TUNED_LEARNING_RATE),
    '--tuned-seed', str(TUNED_SEED),
    '--tuned-schedule', 'plateau',
    '--tuned-loss-normalize',
]

previous_argv = sys.argv
sys.argv = zn_full_training_args
try:
    print('Starting full 9-element pipeline training...')
    pipeline.main()
    print('Full 9-element pipeline training complete.')
finally:
    sys.argv = previous_argv
    pipeline.FEFF_TASKS = original_feff_tasks

if not (ZN_FULL_RUN_DIR / 'RUN_COMPLETE.json').is_file():
    raise FileNotFoundError(f'Full run did not complete: missing {ZN_FULL_RUN_DIR / "RUN_COMPLETE.json"}')
print(f'Full 9-element run complete: {ZN_FULL_RUN_DIR}')

## Evaluate Both Runs on Zn Test Data

Loads the UniversalXAS and tuned Zn checkpoints from both runs and evaluates on the same Zn test targets. Each run uses its own Zn training targets for the train-mean eta baseline.

In [ ]:
from omnixas.model.m3gnet_xas import XASSpectralHead

CU = 'Zn'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
zn_test_ids = read_ids(COMBINED_DATA_ROOT / 'material_id_and_site' / 'Zn_FEFF_test.txt')
zn_test_y = read_targets(COMBINED_DATA_ROOT / 'ml_data' / 'Zn_FEFF_test_y.txt')
zn_train_y_expert = read_targets(COMBINED_DATA_ROOT / 'ml_data' / 'Zn_FEFF_train_y.txt')
zn_train_y_full = read_targets(COMBINED_DATA_ROOT / 'ml_data' / 'Zn_FEFF_train_y.txt')

print(f'Zn test spectra: {len(zn_test_ids)}')
print(f'Expert Zn train spectra: {len(zn_train_y_expert)}')
print(f'Full Zn train spectra: {len(zn_train_y_full)}')

def load_features(run_dir, split):
    path = run_dir / 'features' / f'{CU}_FEFF_{split}_X.txt'
    values = np.loadtxt(path, dtype=np.float32, ndmin=2)
    expected = len(zn_test_ids)
    if values.shape != (expected, 64) or not np.isfinite(values).all():
        raise ValueError(f'Invalid Zn features for {run_dir.name} {split}: {values.shape}')
    return values

def load_head(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing checkpoint: {path}')
    head = XASSpectralHead(dropout=HEAD_DROPOUT).to(device)
    state = torch.load(path, map_location='cpu', weights_only=False)
    head.load_state_dict(state['state_dict'], strict=True)
    return head.eval()

def predict(head, features):
    batches = []
    with torch.inference_mode():
        for start in range(0, len(features), 4096):
            batch = torch.as_tensor(features[start:start + 4096], dtype=torch.float32, device=device)
            batches.append(head(batch).cpu().numpy())
    return np.concatenate(batches, axis=0)

def score(variant, regime, split, targets, predictions, train_targets):
    errors = np.mean((targets - predictions) ** 2, axis=1)
    baseline_errors = np.mean((targets - train_targets.mean(axis=0, keepdims=True)) ** 2, axis=1)
    median_error = float(np.median(errors))
    return {
        'element': CU, 'training_regime': regime, 'variant': variant,
        'evaluation_split': split, 'n_spectra': len(targets),
        'aggregate_mse': float(np.mean(errors)),
        'median_spectrum_mse': median_error,
        'baseline_median_mse': float(np.median(baseline_errors)),
        'eta': float(np.median(baseline_errors) / max(median_error, 1e-12)),
    }, errors

runs = {
    'Expert Zn': (ZN_EXPERT_RUN_DIR, zn_train_y_expert),
    'Full 9-element': (ZN_FULL_RUN_DIR, zn_train_y_full),
}
records, test_predictions = [], {}
for regime, (run_dir, train_targets) in tqdm(runs.items(), desc='Evaluate runs'):
    features = {split: load_features(run_dir, split) for split in ('val', 'test')}
    heads = {
        'UniversalXAS': run_dir / 'heads' / 'universalXAS' / 'best.pt',
        'Tuned-UniversalXAS': run_dir / 'heads' / 'tunedUniversalXAS' / 'Zn_FEFF' / 'best.pt',
    }
    for variant in tqdm(heads, desc=f'{regime} heads', leave=False):
        head = load_head(heads[variant])
        for split in ('val', 'test'):
            targets = zn_test_y if split == 'test' else read_targets(COMBINED_DATA_ROOT / 'ml_data' / f'Zn_FEFF_{split}_y.txt')
            predictions = predict(head, features[split])
            row, errors = score(variant, regime, split, targets, predictions, train_targets)
            records.append(row)
            if split == 'test':
                test_predictions[(regime, variant)] = {'y': targets, 'pred': predictions, 'errors': errors}

results = pd.DataFrame(records)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(ANALYSIS_DIR / 'zn_expert_vs_full_evaluation.csv', index=False)
display(results.round({'aggregate_mse': 6, 'median_spectrum_mse': 6, 'baseline_median_mse': 6, 'eta': 3}))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Comparison bar chart: eta and median MSE for Zn test
test_results = results[results['evaluation_split'] == 'test'].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

x = np.arange(len(test_results))
width = 0.35

axes[0].bar(x - width / 2, test_results['eta'].to_numpy(), width, label='eta')
axes[0].set_xticks(x, [f"{r.training_regime[:6]}\n{r.variant[:3]}" for r in test_results.itertuples()])
axes[0].set_ylabel('eta')
axes[0].set_title('Zn Test Eta: Expert vs Full')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.2)

axes[1].bar(x - width / 2, test_results['median_spectrum_mse'].to_numpy(), width, label='median MSE')
axes[1].set_xticks(x, [f"{r.training_regime[:6]}\n{r.variant[:3]}" for r in test_results.itertuples()])
axes[1].set_ylabel('Median per-spectrum MSE')
axes[1].set_title('Zn Test Median MSE: Expert vs Full')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# Zn test spectra overlay: Expert vs Full vs Target
expert_pred = test_predictions[('Expert Zn', 'Tuned-UniversalXAS')]['pred']
full_pred = test_predictions[('Full 9-element', 'Tuned-UniversalXAS')]['pred']
target = test_predictions[('Expert Zn', 'Tuned-UniversalXAS')]['y']
expert_errors = test_predictions[('Expert Zn', 'Tuned-UniversalXAS')]['errors']
full_errors = test_predictions[('Full 9-element', 'Tuned-UniversalXAS')]['errors']

percentiles = [10, 50, 90, 99]
energy = np.arange(141) * 0.25  # 0.25 eV step above Zn K-edge

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, percentile in zip(axes.flat, percentiles):
    order = np.argsort(expert_errors)
    idx = order[int(round((percentile / 100) * (len(order) - 1)))]
    ax.plot(energy, target[idx], label='target', linewidth=1.6, color='black')
    ax.plot(energy, expert_pred[idx], label='expert Zn train', linewidth=1.2, alpha=0.8)
    ax.plot(energy, full_pred[idx], label='full 9-element train', linewidth=1.2, alpha=0.8)
    material_site = zn_test_ids[idx]
    material_id, site = material_site.rsplit('_', 1)
    ax.set_title(f'{percentile}th expert error (MSE={expert_errors[idx]:.3g})\n{material_id}-{int(site)}')
    ax.grid(alpha=0.2)
axes.flat[0].legend(frameon=False)
fig.supxlabel('Energy above Zn K-edge start (eV)')
fig.supylabel('Unscaled intensity')
fig.suptitle('Zn K-edge XANES test spectra: Expert Zn vs Full 9-element training', y=1.02)
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / 'Zn_expert_vs_full_test_percentiles.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Zn expert vs full comparison saved in {ANALYSIS_DIR}')

In [ ]:
# Summary table
summary = results.pivot_table(
    index=['training_regime', 'variant'],
    columns='evaluation_split',
    values=['eta', 'median_spectrum_mse', 'aggregate_mse'],
)
display(summary.round({'eta': 3, 'median_spectrum_mse': 6, 'aggregate_mse': 6}))

print(f'\nZn K-edge start energy used: {ZN_K_EDGE_START_EV} eV')
print(f'Expert Zn run: {ZN_EXPERT_RUN_DIR}')
print(f'Full 9-element run: {ZN_FULL_RUN_DIR}')
print(f'Combined dataset: {COMBINED_DATA_ROOT}')
print(f'Analysis reports: {ANALYSIS_DIR}')